# Dot Product — Interactive Calculator

Drag the coloured dots to move the tips of the vectors **a** (blue) and **b** (green) anywhere inside the box. Both vectors start at the origin, and tips snap to half-unit steps.

The panel updates live as you drag:

- **a · b** — the dot product,
- **POSITIVE / MINUS / ZERO** — the sign of the dot product,
- **θ** — the angle between the vectors.

**Try this:** drag the tips until the vectors are perpendicular — what does the dot product become? Then make the angle larger than 90° and watch the sign change.

Press the **Run** button (▶) on the cell below to start the calculator.

In [1]:
import uuid

from IPython.display import HTML, display

# A unique id per run, so re-running the cell never clashes with older output
uid = "dp-" + uuid.uuid4().hex[:8]

html = r"""
<div id="__UID__" style="display:flex; flex-wrap:wrap; gap:20px; align-items:flex-start; font-family:system-ui, -apple-system, sans-serif; user-select:none; -webkit-user-select:none;">
  <svg class="dp-svg" width="520" height="520" style="border:1px solid #bbb; background:#fff; touch-action:none;">
    <defs>
      <marker id="__UID__-ma" viewBox="0 0 10 10" refX="7" refY="5" markerWidth="3.5" markerHeight="3.5" orient="auto">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#1f77b4"></path>
      </marker>
      <marker id="__UID__-mb" viewBox="0 0 10 10" refX="7" refY="5" markerWidth="3.5" markerHeight="3.5" orient="auto">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#2ca02c"></path>
      </marker>
    </defs>
    <g class="dp-world" transform="translate(260 260) scale(23.636363636363637 -23.636363636363637)"></g>
  </svg>
  <div class="dp-panel" style="min-width:250px; font-size:15px; line-height:1.7;">
    <div style="font-weight:700;">Drag the coloured dots</div>
    <div><b style="color:#1f77b4;">a</b> = (<span class="dp-ax"></span>, <span class="dp-ay"></span>), |a| = <span class="dp-lena"></span></div>
    <div><b style="color:#2ca02c;">b</b> = (<span class="dp-bx"></span>, <span class="dp-by"></span>), |b| = <span class="dp-lenb"></span></div>
    <hr style="border:none; border-top:1px solid #ddd; margin:10px 0;">
    <div style="color:#555; font-size:13px;">a · b = a<sub>x</sub>b<sub>x</sub> + a<sub>y</sub>b<sub>y</sub></div>
    <div class="dp-expand" style="color:#555; font-size:13px;"></div>
    <div style="font-size:30px; font-weight:800;">a · b = <span class="dp-dot"></span></div>
    <div class="dp-sign" style="display:inline-block; margin-top:2px; padding:4px 16px; border-radius:999px; color:#fff; font-weight:800; letter-spacing:0.1em;"></div>
    <div style="margin-top:10px;">θ = <span class="dp-theta"></span></div>
  </div>
</div>
<script>
(function () {
  var root = document.getElementById('__UID__');
  if (!root && document.currentScript) { root = document.currentScript.previousElementSibling; }
  if (!root) { return; }

  var svg = root.querySelector('.dp-svg');
  var world = root.querySelector('.dp-world');
  var NS = 'http://www.w3.org/2000/svg';
  var HALF = 10;  // the box runs from -10 to +10 on both axes

  function el(name, attrs) {
    var n = document.createElementNS(NS, name);
    for (var k in attrs) { n.setAttribute(k, attrs[k]); }
    world.appendChild(n);
    return n;
  }
  function txt(cls) { return root.querySelector(cls); }

  // Grid, box border, axes, and origin
  var i;
  for (i = -HALF; i <= HALF; i++) {
    el('line', { x1: i, y1: -HALF, x2: i, y2: HALF, stroke: '#e8e8e8', 'stroke-width': 0.045 });
    el('line', { x1: -HALF, y1: i, x2: HALF, y2: i, stroke: '#e8e8e8', 'stroke-width': 0.045 });
  }
  el('rect', { x: -HALF, y: -HALF, width: 2 * HALF, height: 2 * HALF, fill: 'none', stroke: '#999', 'stroke-width': 0.1 });
  el('line', { x1: -HALF, y1: 0, x2: HALF, y2: 0, stroke: '#999', 'stroke-width': 0.1 });
  el('line', { x1: 0, y1: -HALF, x2: 0, y2: HALF, stroke: '#999', 'stroke-width': 0.1 });
  el('circle', { cx: 0, cy: 0, r: 0.16, fill: '#333' });

  // The two vectors
  var a = { x: 6, y: 2 };
  var b = { x: 3, y: 4 };

  var lineA = el('line', { x1: 0, y1: 0, stroke: '#1f77b4', 'stroke-width': 0.24, 'marker-end': 'url(#__UID__-ma)' });
  var lineB = el('line', { x1: 0, y1: 0, stroke: '#2ca02c', 'stroke-width': 0.24, 'marker-end': 'url(#__UID__-mb)' });
  var labA = el('text', { 'font-size': 0.75, fill: '#1f77b4', 'font-weight': 700, 'font-style': 'italic' });
  labA.textContent = 'a';
  var labB = el('text', { 'font-size': 0.75, fill: '#2ca02c', 'font-weight': 700, 'font-style': 'italic' });
  labB.textContent = 'b';
  var dotA = el('circle', { r: 0.42, fill: '#fff', stroke: '#1f77b4', 'stroke-width': 0.18 });
  var dotB = el('circle', { r: 0.42, fill: '#fff', stroke: '#2ca02c', 'stroke-width': 0.18 });
  var hitA = el('circle', { r: 0.9, fill: '#000', 'fill-opacity': 0, 'pointer-events': 'all', style: 'cursor:grab;' });
  var hitB = el('circle', { r: 0.9, fill: '#000', 'fill-opacity': 0, 'pointer-events': 'all', style: 'cursor:grab;' });

  function fmt(v) {
    var r = Math.round(v * 100) / 100;
    if (Object.is(r, -0)) { r = 0; }
    return r.toString();
  }
  function snap(v) { return Math.round(v * 2) / 2; }
  function clamp(v) { return Math.max(-HALF, Math.min(HALF, v)); }

  function render() {
    lineA.setAttribute('x2', a.x); lineA.setAttribute('y2', a.y);
    lineB.setAttribute('x2', b.x); lineB.setAttribute('y2', b.y);
    dotA.setAttribute('cx', a.x); dotA.setAttribute('cy', a.y);
    dotB.setAttribute('cx', b.x); dotB.setAttribute('cy', b.y);
    hitA.setAttribute('cx', a.x); hitA.setAttribute('cy', a.y);
    hitB.setAttribute('cx', b.x); hitB.setAttribute('cy', b.y);
    labA.setAttribute('transform', 'translate(' + (a.x + 0.55) + ' ' + (a.y + 0.35) + ') scale(1 -1)');
    labB.setAttribute('transform', 'translate(' + (b.x + 0.55) + ' ' + (b.y + 0.35) + ') scale(1 -1)');

    var dot = a.x * b.x + a.y * b.y;
    var lenA = Math.hypot(a.x, a.y);
    var lenB = Math.hypot(b.x, b.y);

    txt('.dp-ax').textContent = fmt(a.x);
    txt('.dp-ay').textContent = fmt(a.y);
    txt('.dp-bx').textContent = fmt(b.x);
    txt('.dp-by').textContent = fmt(b.y);
    txt('.dp-lena').textContent = fmt(lenA);
    txt('.dp-lenb').textContent = fmt(lenB);
    txt('.dp-expand').textContent = '= (' + fmt(a.x) + ')(' + fmt(b.x) + ') + (' + fmt(a.y) + ')(' + fmt(b.y) + ')';
    txt('.dp-dot').textContent = fmt(dot);

    var sign = txt('.dp-sign');
    if (Math.abs(dot) < 1e-9) {
      sign.textContent = 'ZERO';
      sign.style.background = '#777';
    } else if (dot > 0) {
      sign.textContent = 'POSITIVE';
      sign.style.background = '#2ca02c';
    } else {
      sign.textContent = 'MINUS';
      sign.style.background = '#d62728';
    }

    if (lenA < 1e-9 || lenB < 1e-9) {
      txt('.dp-theta').textContent = '—';
    } else {
      var c = Math.max(-1, Math.min(1, dot / (lenA * lenB)));
      txt('.dp-theta').textContent = (Math.acos(c) * 180 / Math.PI).toFixed(1) + '°';
    }
  }

  var dragging = null;
  function toWorld(evt) {
    var p = new DOMPoint(evt.clientX, evt.clientY);
    return p.matrixTransform(world.getScreenCTM().inverse());
  }
  function makeDraggable(hit, vec) {
    hit.addEventListener('pointerdown', function (e) {
      dragging = vec;
      hit.setPointerCapture(e.pointerId);
      hit.style.cursor = 'grabbing';
      e.preventDefault();
    });
    hit.addEventListener('pointermove', function (e) {
      if (dragging !== vec) { return; }
      var p = toWorld(e);
      vec.x = clamp(snap(p.x));
      vec.y = clamp(snap(p.y));
      render();
    });
    function stop() {
      if (dragging === vec) { dragging = null; hit.style.cursor = 'grab'; }
    }
    hit.addEventListener('pointerup', stop);
    hit.addEventListener('pointercancel', stop);
  }
  makeDraggable(hitA, a);
  makeDraggable(hitB, b);
  render();
})();
</script>
""".replace("__UID__", uid)

display(HTML(html))

## Maths used

$$\mathbf{a}\cdot\mathbf{b} = a_x b_x + a_y b_y = |\mathbf{a}|\,|\mathbf{b}|\cos\theta$$

The sign of the dot product tells you the angle between the vectors:

| a · b | angle θ | meaning |
|---|---|---|
| POSITIVE | less than 90° | vectors point in similar directions |
| ZERO | exactly 90° | vectors are perpendicular (or one has zero length) |
| MINUS | more than 90° | vectors point in opposing directions |